---
title: "Download: `download` Module as a `pytask` Task"
execute:
  freeze: auto
engine: jupyter
---

## task_download 

> This module downloads the raw era5 data from the CDS API. It is similar to the original script, refactored for `pytask`.

In [ ]:
#| default_exp task_download:
#|

In [ ]:
#| hide:
# showdoc
from nbdev.showdoc import *

We're going to quickly refactor the pipeline to use pytask instead of hydra and snakemake. This will hopefully demonstrate a simpler and more flexible way to manage data pipelines in Python.

To start off, we need to create a function that queries the CDS API with one job. This function will be used to download the data for each query in the range specified in the data catalog in the config file.

Let's take a look at the data catalog we created in the config module:

In [ ]:
#| export:
# necessary imports
import cdsapi
import pytask
import os
from pytask import task, Product
from pathlib import Path
from typing import Annotated
from pandas import Series

from era5_sandbox.config import data_catalog
from era5_sandbox.config import BLD
from era5_sandbox.config import DEV_MODE
from era5_sandbox.pytask_logger import setup_logger
from era5_sandbox.download import fetch_GADM, create_bounding_box

You can see the queries entry we created in the data catalog. Each query is a row of a dataframe that contains the parameters for the CDS API query.

In [ ]:
queries = data_catalog['download']['jobs']['queries_df'].load()
queries

We can test this query like we did in the original work:

In [ ]:
example_query = queries.iloc[0]

create_bounding_box(example_query['shapefile'])

In this way, we have a similar approach as Hydra configs, but, using the `pytask` data catalog, we can more easily gather the data for a specific task in structured manner entirely in Python.

In [ ]:
#| eval: false

client = cdsapi.Client()

ex_bounding_box = create_bounding_box(example_query['shapefile'])

request = {
            "product_type": example_query['product_type'],
            "variable": example_query['variables'], 
            "year": str(example_query['year']),
            "month": str(example_query['month']),
            "day": example_query['day'],
            "time": example_query['time'],
            "data_format": "netcdf",
            "download_format": "unarchived",
            "area": ex_bounding_box
        }

target = f"{example_query['output']}.nc"

client.retrieve("reanalysis-era5-single-levels", request).download(target)

This works! So now we just need to create a `task_` function that pytask will recognise to parallelise the download of queries over:

In [ ]:
#| export:
# define the download task

queries = data_catalog['download']['jobs']['queries_df'].load()

for i, job in queries.iterrows():

    @task(id=job['output'], name=f"Download {job['output']}")
    def task_download_raw_data(
        _query: Series = job   # The query object from the data catalog
    )-> Annotated[Path, data_catalog['download']['outputs'][job['output']]]:
        
        logger = setup_logger(_query['output'])
        output_path = BLD / f"{_query['output']}.nc"
        logger.info(f"Starting download for {_query['output']} to {output_path}")

        # check if string file path exists
        if os.path.exists(output_path):
            logger.info(f"File {output_path} already exists. Skipping download.")
            return output_path

        client = cdsapi.Client()
        bounding_box = create_bounding_box(_query['shapefile'])
    
        request = {
                "product_type": _query['product_type'],
                "variable": _query['variables'], 
                "year": _query['year'],
                "month": _query['month'],
                "day": _query['day'],
                "time": _query['time'],
                "data_format": "netcdf",
                "download_format": "unarchived",
                "area": bounding_box
            }
                
        client.retrieve("reanalysis-era5-land", request).download(output_path)
        logger.info(f"Downloaded data for {_query['output']} to {output_path}")

        return output_path

### How this works (with some help from GPT):

#### 🧠 How pytask Discovers and Executes Tasks

When you run pytask, it automatically scans your project for Python files named `task_*.py`. In these files, it looks for:
- Functions decorated with `@task`, or
- Functions prefixed with `task_`

These functions are not executed immediately. Instead, `pytask`:
1.	Imports each task_*.py module (just like Python would)
2.	Registers any matching task functions as nodes in a directed acyclic graph (DAG)
3.	Resolves dependencies by analyzing:
    - Input annotations (e.g., `Annotated[x, DependsOn]`)
    - Output declarations (e.g., `return` values or `Product` annotations)
4.	Builds the DAG, where each task function is a node
5.	Executes the tasks, respecting dependency order and skipping up-to-date nodes

So even though the task functions aren’t explicitly “run” in the Python code itself, pytask knows how and when to execute them — based on their position in the DAG.

#### 🔄 How This Differs from Snakemake

In `snakemake`, you’re expected to define a series of explicitly executable rules, often using shell commands or Python scripts. You “stitch together” rules using filenames and wildcard matching.

In contrast:
- 🐍 pytask is Python-native — tasks are just regular Python functions
- ⚙️ It builds a DAG from those functions and tracks inputs/outputs automatically
- 🧱 You are declaring nodes, not scripting execution

Think of your Python files not as scripts to run, but as a way to define and wire together declarative tasks that will be executed by the pytask engine.

---

Because we defined this task in a function and loop, we can easily debug a node in the DAG by simply calling it:

In [ ]:
#| eval: false
task_download_raw_data()